# 02 Data Cleaning
Clean text data, handle missing values, remove duplicates, and prepare for analysis.

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from pathlib import Path

# Download NLTK data
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

# Paths
DATA_PATH = '../data/Amazon_Reviews.csv'
OUTPUT_PATH = '../data/cleaned_reviews.csv'

print('Loading raw data...')
df = pd.read_csv(DATA_PATH)
print(f'Initial shape: {df.shape}')

In [ ]:
# Text cleaning function
def clean_text(text):
    if pd.isna(text):
        return ''
    # lowercase
    text = str(text).lower()
    # remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # remove HTML entities
    text = re.sub(r'&[a-z]+;', '', text)
    # remove special characters, keep only letters and spaces
    text = re.sub(r'[^a-z\s]', ' ', text)
    # remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # remove stopwords and short tokens
    tokens = [w for w in text.split() if w not in stop_words and len(w) > 2]
    return ' '.join(tokens)

print('Cleaning text...')
if 'reviewText' in df.columns:
    df['reviewText_clean'] = df['reviewText'].apply(clean_text)
    df = df[df['reviewText_clean'].str.strip().astype(bool)].reset_index(drop=True)
    
print(f'After cleaning: {df.shape}')

In [ ]:
# Remove duplicates
print('Removing duplicates...')
initial = len(df)
df = df.drop_duplicates(subset=['reviewText_clean'], keep='first').reset_index(drop=True)
removed = initial - len(df)
print(f'Removed {removed} duplicates. Shape: {df.shape}')

In [ ]:
# Handle missing values
print('Handling missing values...')
if 'summary' in df.columns:
    df['summary'] = df['summary'].fillna('No summary')
if 'overall' in df.columns:
    df['overall'] = df['overall'].fillna(df['overall'].median())

# Create sentiment label (binary)
if 'overall' in df.columns:
    df['sentiment'] = (df['overall'] >= 4).astype(int)
    print('\nSentiment distribution:')
    print(df['sentiment'].value_counts())

print(f'\nFinal shape: {df.shape}')

In [ ]:
# Save cleaned dataset
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print(f'Cleaned data saved to {OUTPUT_PATH}')

# Display sample
df[['reviewText_clean', 'overall', 'sentiment']].head(10)